In [1]:
FOUNDRY_PROJECT_ENDPOINT="https://foundry-azd-web-agents-luna.services.ai.azure.us/api/projects/web-agents"
FOUNDRY_AGENT_NAME="test-agent"
FOUNDRY_AGENT_VERSION="1"

In [3]:
from azure.identity import DefaultAzureCredential, AzureAuthorityHosts, get_bearer_token_provider
from azure.ai.projects import AIProjectClient

endpoint = "https://foundry-azd-web-agents-luna.services.ai.azure.us/api/projects/web-agents"
my_agent = "test-agent"
my_version = "1"

project_client = AIProjectClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(authority=AzureAuthorityHosts.AZURE_GOVERNMENT),
)

openai_client = project_client.get_openai_client(
    api_key=get_bearer_token_provider(
        project_client._config.credential,
        "https://ai.azure.us/.default",
    )
)

# Reference the agent to get a response
response = openai_client.responses.create(
    input=[{"role": "user", "content": "Tell me what you can help with."}],
    extra_body={"agent_reference": {"name": my_agent, "version": my_version, "type": "agent_reference"}},
)

print(f"Response output: {response.output_text}")

Response output: I can help with things like:

- Answering questions and explaining concepts
- Writing, editing, summarizing, and translating text
- Brainstorming ideas, names, plans, and creative content
- Coding, debugging, and explaining technical topics
- Analyzing data, documents, or images you provide
- Planning trips, projects, study schedules, or workflows
- Preparing resumes, cover letters, emails, presentations, and reports
- Practicing interviews, conversations, or languages
- Comparing options and helping you make decisions

Tell me what you’re working on, and I’ll help you figure it out.


In [7]:
from agent_framework.foundry import FoundryAgent
from azure.ai.projects.aio import AIProjectClient as AsyncAIProjectClient
from azure.identity import AzureAuthorityHosts
from azure.identity.aio import (
    DefaultAzureCredential as AsyncDefaultAzureCredential,
    get_bearer_token_provider as get_async_bearer_token_provider,
)


class GovernmentAIProjectClient(AsyncAIProjectClient):
    def __init__(self, *, endpoint, credential):
        scope = "https://ai.azure.us/.default"
        super().__init__(
            endpoint=endpoint, credential=credential, credential_scopes=[scope]
        )
        self._government_token_provider = get_async_bearer_token_provider(
            credential, scope
        )

    def get_openai_client(self, **kwargs):
        kwargs.setdefault("api_key", self._government_token_provider)
        return super().get_openai_client(**kwargs)


async with AsyncDefaultAzureCredential(
    authority=AzureAuthorityHosts.AZURE_GOVERNMENT
) as credential:
    async with GovernmentAIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential
    ) as government_project_client:
        async with FoundryAgent(
            project_client=government_project_client,
            agent_name=FOUNDRY_AGENT_NAME,
            agent_version=FOUNDRY_AGENT_VERSION,
        ) as agent:
            result = await agent.run("Summarize the latest news about AI.")
            print(f"Agent: {result}")

Agent: I don’t have live news access, so I can’t reliably summarize developments from today. My latest reliable information is from **June 2024**. The major AI news then included:

- **OpenAI launched GPT‑4o**, a faster multimodal model capable of real-time voice, vision, and text interaction; some features were offered free.
- **Anthropic released Claude 3.5 Sonnet**, reporting major gains in coding, reasoning, and visual tasks.
- **Google expanded Gemini**, including Gemini 1.5 Pro/Flash and AI-generated summaries in Google Search, prompting concerns about accuracy and publisher traffic.
- **Apple announced Apple Intelligence**, integrating on-device and cloud AI into iPhones, iPads, and Macs, with ChatGPT integration.
- **Microsoft promoted Copilot+ PCs**, built around new AI-focused processors and features such as Recall—though Recall faced privacy criticism and deployment delays.
- **The EU formally approved the AI Act**, establishing risk-based regulation and transparency require

In [8]:
session = await agent.create_conversation()

first = await agent.run("My name is Blaine.", session=session)
second = await agent.run("What is my name?", session=session)
print(second)

Your name is Blaine.
